### Run inference

Required inputs:

file_name = "interactions_output_file.parquet" <- parquet file containing all interactions

directory = "biointeract100" <- directory where all 

In [ ]:
input_file = "interactions_output_file.parquet" 
directory = "biointeract100"

Libraries:

In [ ]:
import os
import torch
from transformers import AutoModel, AutoProcessor
import open_clip
import pandas as pd
import calendar
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import ast
import numpy as np
from sklearn.metrics import f1_score, recall_score
from itertools import combinations

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class SigLIP2Wrapper(torch.nn.Module):
    def __init__(self, model_name="google/siglip2-base-patch16-224", device="cpu"):
        super().__init__()
        self.model = AutoModel.from_pretrained(model_name)
        self.processor = AutoProcessor.from_pretrained(model_name, use_fast=True)
        self.device = device

    def to(self, device):
        self.device = device
        self.model = self.model.to(device)
        return self

    def eval(self):
        self.model.eval()
        return self

    def encode_image(self, pixel_values):
        return self.model.get_image_features(pixel_values=pixel_values)

    def encode_text(self, input_ids=None, attention_mask=None, **kwargs):
        return self.model.get_text_features(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )


class SigLIP2Preprocess:
    def __init__(self, image_processor):
        self.image_processor = image_processor

    def __call__(self, image):
        out = self.image_processor(images=image, return_tensors="pt")
        return out["pixel_values"].squeeze(0)

class SigLIP2Tokenizer:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        return self.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=False,
        )

def create_siglip2_model_and_transforms(
    model_name="google/siglip2-base-patch16-224",
    device="cpu",
):
    model = SigLIP2Wrapper(model_name=model_name, device=device)
    preprocess = SigLIP2Preprocess(model.processor.image_processor)
    tokenizer = SigLIP2Tokenizer(model.processor.tokenizer)
    return model, preprocess, preprocess, tokenizer

In [ ]:
def compute_consistency_at_k(rankings_by_task, k=50):

    pairwise_scores = {}

    if len(rankings_by_task) < 2:
        return {
            "Consistency@50": np.nan,
            "Consistency@50_std": np.nan,
            "Consistency@50_min": np.nan,
            "pairwise_consistency": pairwise_scores,
        }

    for t1, t2 in combinations(rankings_by_task.keys(), 2):
        top1 = set(rankings_by_task[t1][:k])
        top2 = set(rankings_by_task[t2][:k])

        denom = min(k, len(top1), len(top2))
        if denom == 0:
            continue

        pairwise_scores[(t1, t2)] = len(top1 & top2) / denom

    values = list(pairwise_scores.values())

    if len(values) == 0:
        return {
            "Consistency@50": np.nan,
            "Consistency@50_std": np.nan,
            "Consistency@50_min": np.nan,
            "pairwise_consistency": pairwise_scores,
        }

    return {
        "Consistency@50": float(np.mean(values)),
        "Consistency@50_std": float(np.std(values)),
        "Consistency@50_min": float(np.min(values)),
        "pairwise_consistency": pairwise_scores,
    }

def average_precision_at_k(relevance, total_relevant, k=50):
    """
    relevance: list of 0/1 values in ranked order over the full ranking
    total_relevant: total number of relevant items for this query in the dataset
    returns AP@k
    """
    relevance_k = relevance[:k]

    if total_relevant == 0:
        return 0.0

    hits = 0
    ap = 0.0

    for rank, rel in enumerate(relevance_k, start=1):
        if rel:
            hits += 1
            ap += hits / rank

    return ap / min(total_relevant, k)


def reciprocal_rank(relevance):
    """
    relevance: list of 0/1 values in ranked order
    returns reciprocal rank (RR)
    """
    for rank, rel in enumerate(relevance, start=1):
        if rel:
            return 1.0 / rank
    return 0.0


def paired_bootstrap_ci(forward_scores, reverse_scores, n_boot=10000, alpha=0.05, seed=42):
    """
    Computes paired bootstrap confidence interval for the mean difference:
        delta = mean(forward_scores - reverse_scores)

    Args:
        forward_scores: list or array of per-query metric values
        reverse_scores: list or array of per-query metric values
        n_boot: number of bootstrap samples
        alpha: significance level (0.05 -> 95% CI)
        seed: random seed

    Returns:
        dict with mean delta and confidence interval
    """
    forward_scores = np.asarray(forward_scores, dtype=float)
    reverse_scores = np.asarray(reverse_scores, dtype=float)

    if len(forward_scores) != len(reverse_scores):
        raise ValueError("forward_scores and reverse_scores must have the same length")

    if len(forward_scores) == 0:
        raise ValueError("Score lists must not be empty")

    diffs = forward_scores - reverse_scores
    n = len(diffs)

    rng = np.random.default_rng(seed)
    boot_means = np.empty(n_boot, dtype=float)

    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_means[i] = diffs[idx].mean()

    mean_delta = diffs.mean()
    lower = np.percentile(boot_means, 100 * (alpha / 2))
    upper = np.percentile(boot_means, 100 * (1 - alpha / 2))

    return {
        "delta_mean": float(mean_delta),
        "ci_lower": float(lower),
        "ci_upper": float(upper),
    }


def compute_retrieval_metrics(
    similarity,
    gt_per_image,
    candidate_texts,
    vlm,
    task,
    ap_k=50,
    recall_k=10,
    return_per_query=False,
):
    """
    similarity: [num_images, num_texts]
        similarity[i, j] = sim(image_i, text_j)

    gt_per_image: list of labels, one ground-truth label per image

    candidate_texts: list of text queries corresponding to columns of similarity

    Returns macro-averaged metrics over text queries.
    If return_per_query=True, also returns per-query scores.
    """
    ap_scores = []
    rr_scores = []
    recall_scores = []

    for text_idx, query_text in enumerate(candidate_texts):
        sims = similarity[:, text_idx]
        ranked_img_idx = torch.argsort(sims, descending=True)

        ranked_labels = [gt_per_image[i] for i in ranked_img_idx.tolist()]
        relevance = [1 if label == query_text else 0 for label in ranked_labels]

        total_relevant = sum(1 for label in gt_per_image if label == query_text)

        ap = average_precision_at_k(
            relevance=relevance,
            total_relevant=total_relevant,
            k=ap_k,
        )
        rr = reciprocal_rank(relevance)
        recall = 1.0 if any(relevance[:recall_k]) else 0.0

        ap_scores.append(ap)
        rr_scores.append(rr)
        recall_scores.append(recall)

    results = {
        "vlm": vlm,
        "task": task,
        f"AP@{ap_k}": float(np.mean(ap_scores)),
        "MRR": float(np.mean(rr_scores)),
        f"Recall@{recall_k}": float(np.mean(recall_scores)),
    }

    if return_per_query:
        results["per_query"] = {
            f"AP@{ap_k}": ap_scores,
            "MRR": rr_scores,
            f"Recall@{recall_k}": recall_scores,
        }

    return results


def compute_directionality_with_bootstrap(
    forward_similarity,
    reverse_similarity,
    gt_per_image,
    candidate_texts1,
    candidate_texts2,
    vlm,
    task,
    ap_k=50,
    recall_k=10,
    n_boot=10000,
    alpha=0.05,
    seed=42,
):
    """
    Computes forward and reverse retrieval metrics, then estimates paired bootstrap
    confidence intervals for delta AP@k and delta MRR.

    Delta is defined as:
        forward - reverse
    """
    forward_results = compute_retrieval_metrics(
        similarity=forward_similarity,
        gt_per_image=gt_per_image,
        candidate_texts=candidate_texts1,
        vlm=vlm,
        task=f"{task}_forward",
        ap_k=ap_k,
        recall_k=recall_k,
        return_per_query=True,
    )

    reverse_results = compute_retrieval_metrics(
        similarity=reverse_similarity,
        gt_per_image=gt_per_image,
        candidate_texts=candidate_texts2,
        vlm=vlm,
        task=f"{task}_reverse",
        ap_k=ap_k,
        recall_k=recall_k,
        return_per_query=True,
    )

    ap_key = f"AP@{ap_k}"
    recall_key = f"Recall@{recall_k}"

    delta_ap = paired_bootstrap_ci(
        forward_results["per_query"][ap_key],
        reverse_results["per_query"][ap_key],
        n_boot=n_boot,
        alpha=alpha,
        seed=seed,
    )

    delta_mrr = paired_bootstrap_ci(
        forward_results["per_query"]["MRR"],
        reverse_results["per_query"]["MRR"],
        n_boot=n_boot,
        alpha=alpha,
        seed=seed,
    )

    delta_recall = paired_bootstrap_ci(
        forward_results["per_query"][recall_key],
        reverse_results["per_query"][recall_key],
        n_boot=n_boot,
        alpha=alpha,
        seed=seed,
    )

    return {
        "vlm": vlm,
        "task": task,
        "forward": {
            ap_key: forward_results[ap_key],
            "MRR": forward_results["MRR"],
            recall_key: forward_results[recall_key],
        },
        "reverse": {
            ap_key: reverse_results[ap_key],
            "MRR": reverse_results["MRR"],
            recall_key: reverse_results[recall_key],
        },
        "delta": {
            ap_key: delta_ap,
            "MRR": delta_mrr,
            recall_key: delta_recall,
        },
    }

In [265]:
def get_model_tokenizer(vlm):

    if vlm == "siglip2":
        # Load the SigLip2 as CLIP-like 
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            "ViT-L-16-SigLIP2-256",
            pretrained="webli"
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer("ViT-L-16-SigLIP2-256")

    if vlm == 'siglip':
        # Load the SigLip
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            "ViT-SO400M-14-SigLIP",
            pretrained="webli",
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer("ViT-SO400M-14-SigLIP")
    
    if vlm == 'bioclip2':
        # Load the BioCLIP 2 model from Hugging Face
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            'hf-hub:imageomics/bioclip-2'
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer('hf-hub:imageomics/bioclip-2')

    if vlm == "bioclip":
        # Load the BioCLIP model from Hugging Face
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            'hf-hub:imageomics/bioclip'
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer('hf-hub:imageomics/bioclip')
    
    if vlm == "clip":
        # Load the CLIP model from OpenAI
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            'ViT-L-14',
            pretrained='openai'
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer('ViT-L-14')

    if vlm == "metaclip":
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            "ViT-L-14-quickgelu",
            pretrained="metaclip_fullcc"
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer("ViT-L-14-quickgelu")

    if vlm == "taxabind":
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            "hf-hub:MVRL/taxabind-vit-b-16"
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer("hf-hub:MVRL/taxabind-vit-b-16")

    if vlm == "biocap":
        # Load the BioCAP model from Hugging Face
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            "hf-hub:imageomics/biocap"
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer("hf-hub:imageomics/biocap")

    if "biotrove" in vlm:
        if "biotrove-bioclip" in vlm:
            model_name = "hf-hub:imageomics/bioclip"
            ckpt_path = "../biotrove-clip/biotroveclip-vit-b-16-from-bioclip-epoch-8.pt"
        if "biotrove-metaclip" in vlm:
            model_name = "ViT-L-14"
            ckpt_path = "../biotrove-clip/biotroveclip-vit-l-14-from-metaclip-epoch-12.pt"
        if "biotrove-openai" in vlm:
            model_name = "ViT-B-16"
            ckpt_path = "../biotrove-clip/biotroveclip-vit-b-16-from-openai-epoch-40.pt"

        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            model_name,
            pretrained=None,
        )
        tokenizer = open_clip.get_tokenizer(model_name)
        model = model.to(device).eval()

    return model, tokenizer

In [ ]:
def load_data(folder, file, input_file, interact=False):
    img_feat_data = torch.load(f"{folder}{file}", weights_only=False)

    img_feat = img_feat_data["emb"]
    df = img_feat_data["df"]

    parquet_file = input_file
    
    df_benchmark = pd.read_parquet(parquet_file,     
                        engine="pyarrow",
                        dtype_backend="pyarrow"
    )
    
    # filter based on benchmark
    keys = ["sourceTaxonName", "targetTaxonName", "interactionTypeName"]    
    mask = df.set_index(keys).index.isin(
        df_benchmark.set_index(keys).index
    )
    df_filtered = df[mask]

    if interact == True:
        df_filtered = df_filtered[df_filtered["interactionTypeName"] != "interactsWith"]
    
    # get right index
    idx = df_filtered.index
    img_feat = img_feat[idx]
    df = df_filtered
    df = df.reset_index(drop=True)

    return img_feat, df

Prepare text:

In [ ]:
def to_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, tuple):
        return list(x)
    if isinstance(x, np.ndarray):
        return x.tolist()

    if x is None:
        return []

    if isinstance(x, str):
        x = x.strip()
        if x == "":
            return []
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
            if isinstance(parsed, tuple):
                return list(parsed)
            return [parsed]
        except Exception:
            return [x]

    if pd.isna(x):
        return []

    return [x]

def get_labels(task, df, interact):

    df_search = pd.read_parquet("../../data/benchmarks/classification_balanced_unique_benchmark_biointeractseen.parquet",
            engine="pyarrow",
            dtype_backend="pyarrow"
        )
    if interact is True:
        df_search = df_search[df_search["interactionTypeName"] != "interactsWith"]

    print(len(df_search))

    df_search["interactionTypeName_normal"] = (
        df_search["interactionTypeName"]
        .astype("string[python]")
        .str.replace(r"([a-z])([A-Z])", r"\1 \2", regex=True)
        .str.lower()
        .str.strip()
        )

    reverse_map = {
        "visits": "is visited by",
        "visits flowers of": "is visited by",
        "eats": "is eaten by",
        "preys on": "is preyed on by",
        "has host": "is hosted by",
        "parasite of": "is parasitized by",
        "pathogen of": "has pathogen",
        "parasitoid of": "has parasitoid",
        "interacts with": "interacts with",
    }

    # Normalize interaction names
    df["interactionTypeName_normal"] = (
        df["interactionTypeName"]
        .astype("string[python]")
        .str.replace(r"([a-z])([A-Z])", r"\1 \2", regex=True)
        .str.lower()
        .str.strip()
    )
    taxon_map = {
    "Animalia": "animal",
    "Plantae": "plant",
    "Fungi": "fungus",
    "Insecta": "insect",
    "Aves": "bird",
    "Mammalia": "mammal",
    }

    if task == "active":
        df["ground_truth"]= df["sourceTaxonName"] + " " + df["interactionTypeName_normal"] + " " + df["targetTaxonName"]
        candidate_texts = df_search["sourceTaxonName"] + " " + df_search["interactionTypeName_normal"] + " " + df_search["targetTaxonName"]

    if task == "target":
        df["interactionTypeName_passive"] = df["interactionTypeName_normal"].map(reverse_map)
        df["ground_truth"]= df["targetTaxonName"] + " "+ df["interactionTypeName_passive"] + " another organism"
        df_search["interactionTypeName_passive"]= "" + df_search["interactionTypeName_normal"].map(reverse_map)
        candidate_texts = df_search["targetTaxonName"] + " " + df_search["interactionTypeName_passive"] + " another organism"
    
    if task == "source":
        df["ground_truth"]= df["sourceTaxonName"] + " " + df["interactionTypeName_normal"] + " another organism"
        candidate_texts = df_search["sourceTaxonName"] + " " + df_search["interactionTypeName_normal"] + " another organism"

    if task == "passive":
        df["interactionTypeName_passive"] = df["interactionTypeName_normal"].map(reverse_map)
        df["ground_truth"]= df["targetTaxonName"] + " " + df["interactionTypeName_passive"] + " " + df["sourceTaxonName"]
        df_search["interactionTypeName_passive"]= "" + df_search["interactionTypeName_normal"].map(reverse_map)
        candidate_texts = df_search["targetTaxonName"] + " " + df_search["interactionTypeName_passive"] + " " + df_search["sourceTaxonName"]

    if task == "reverse":
        df["ground_truth"]= "" + df["sourceTaxonName"] + " " + df["interactionTypeName_normal"] + " " + df["targetTaxonName"]
        df_search["ground_truth"]= df_search["sourceTaxonName"] + " " + df_search["interactionTypeName_normal"] + " " + df_search["targetTaxonName"]
        df_search["other"]= df_search["targetTaxonName"] + " " + df_search["interactionTypeName_normal"] + " " + df_search["sourceTaxonName"]
        candidate_texts = pd.unique(df_search[["ground_truth","other"]].values.ravel()).tolist()
        
    if task == "passive_reverse":
        df["interactionTypeName_passive"] = df["interactionTypeName_normal"].map(reverse_map)
        df["ground_truth"] = df["targetTaxonName"] + " " + df["interactionTypeName_passive"] + " " + df["sourceTaxonName"]
        df_search["interactionTypeName_passive"] = df_search["interactionTypeName_normal"].map(reverse_map)
        df_search["ground_truth"]= "" + df_search["targetTaxonName"] + " " + df_search["interactionTypeName_passive"] + " " + df_search["sourceTaxonName"]
        df_search["other"]= "" + df_search["sourceTaxonName"] + " " + df_search["interactionTypeName_passive"] + " " + df_search["targetTaxonName"]
        candidate_texts = pd.unique(df_search[["ground_truth","other"]].values.ravel()).tolist()
        
    if task == "no_relation":
        df["ground_truth"]= "" + df["sourceTaxonName"] + " " + df["targetTaxonName"]
        candidate_texts = df_search["sourceTaxonName"] + " " + df_search["targetTaxonName"] 
       
    return df, candidate_texts


In [ ]:
def to_list_of_strings(x):
    if isinstance(x, list):
        return [str(v) for v in x]

    if isinstance(x, np.ndarray):
        return [str(v) for v in x.tolist()]

    if isinstance(x, str):
        x = x.strip()
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return [str(v) for v in parsed]
            if isinstance(parsed, np.ndarray):
                return [str(v) for v in parsed.tolist()]
            return [str(parsed)]
        except (ValueError, SyntaxError):
            return [x]

    if x is None:
        return []

    if pd.isna(x):
        return []

    return [str(x)]

In [ ]:
vlms = ["bioclip", "bioclip2", "siglip", "siglip2", "clip", "metaclip", "biocap", "taxabind", "biotrove-bioclip", "biotrove-openai", "biotrove-metaclip"]

# Collect valid files first
files = [
    f for f in os.listdir(directory)
    if os.path.isfile(os.path.join(directory, f))
]
results = []

tasks = ["active", "source", "target", "passive", "no_relation", "reverse", "passive_reverse"]

for file in files:

    print(file)

    img_feat, df = load_data(directory, file, input_file, interact=True) 

    batch_size = 32

    for vlm in vlms:

        if f"_{vlm}_" not in file:
            continue

        model, tokenizer = get_model_tokenizer(vlm)

        rankings_by_task = {}
        ap_by_task = {}

        for task in tasks:

            df, candidate_texts = get_labels(task, df, interact=True)
            print(len(candidate_texts))

            gt_per_image = df["ground_truth"].tolist()

            with torch.no_grad():

                img_feat_tensor = torch.as_tensor(
                    img_feat, dtype=torch.float32, device=device
                )
                img_feat_tensor = img_feat_tensor / img_feat_tensor.norm(
                    dim=-1, keepdim=True
                )

                text_tokens = tokenizer(candidate_texts).to(device)
                text_feat = model.encode_text(text_tokens)
                text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)

            # image-to-text similarity: N images x K text queries
            similarity = img_feat_tensor @ text_feat.T

            ap_k = 50

            result = compute_retrieval_metrics(
                similarity,
                gt_per_image,
                candidate_texts,
                vlm,
                task,
                ap_k = ap_k
            )

            results.append(result)
            ap_by_task[task] = result[f"AP@{ap_k}"]

            # text-to-image ranking
            # shape: K queries x N images
            text_to_image_similarity = similarity.T

            # If each task has one query, use query index 0.
            # If multiple candidate texts exist, average their top-k overlap separately.
            topk = torch.topk(
                text_to_image_similarity,
                k=min(50, text_to_image_similarity.shape[1]),
                dim=1
            ).indices

            # Store top-50 for each query in this task
            rankings_by_task[task] = topk[0].detach().cpu().tolist()

        consistency = compute_consistency_at_k(rankings_by_task, k=50)

        variant_summary = {
            "vlm": vlm,
            "file": file,
            "task": "all_variants",
            "mean_AP@50": float(np.mean(list(ap_by_task.values()))),
            "std_AP@50": float(np.std(list(ap_by_task.values()))),
            "min_AP@50": float(np.min(list(ap_by_task.values()))),
            "max_AP@50": float(np.max(list(ap_by_task.values()))),
            "delta_AP@50": float(np.max(list(ap_by_task.values())) - np.min(list(ap_by_task.values()))),
            "Consistency@50": consistency["Consistency@50"],
            "Consistency@50_std": consistency["Consistency@50_std"],
            "Consistency@50_min": consistency["Consistency@50_min"],
        }

        results.append(variant_summary)


In [301]:
df_results = pd.DataFrame(results)

df_results.to_csv("retrival_reverse.csv", index=False)

results

[{'vlm': 'bioclip',
  'task': 'reverse',
  'AP@50': 0.19989707231097295,
  'MRR': 0.3757525304739671,
  'Recall@10': 0.46153846153846156},
 {'vlm': 'bioclip',
  'task': 'passive_reverse',
  'AP@50': 0.19925854617695343,
  'MRR': 0.35828738300643814,
  'Recall@10': 0.46474358974358976},
 {'vlm': 'bioclip',
  'file': 'image_embeddings_bioclip_biointeract_benchmark.pt',
  'task': 'all_variants',
  'mean_AP@50': 0.19957780924396318,
  'std_AP@50': 0.00031926306700975904,
  'min_AP@50': 0.19925854617695343,
  'max_AP@50': 0.19989707231097295,
  'delta_AP@50': 0.0006385261340195181,
  'Consistency@50': 0.56,
  'Consistency@50_std': 0.0,
  'Consistency@50_min': 0.56},
 {'vlm': 'bioclip2',
  'task': 'reverse',
  'AP@50': 0.3268575210515544,
  'MRR': 0.44060757529507527,
  'Recall@10': 0.49038461538461536},
 {'vlm': 'bioclip2',
  'task': 'passive_reverse',
  'AP@50': 0.28917516999483267,
  'MRR': 0.41772301939551215,
  'Recall@10': 0.483974358974359},
 {'vlm': 'bioclip2',
  'file': 'image_embed